# C3 — Contrastive Hebbian Attention (CHA)

Forward-Forward'ın attention versiyonu. h⁺/h⁻ farkı routing skoru; lokal Hebbian update key projection ağırlıklarına.

**Bağımsız çalışır.** Tüm parametreler Hücre 2'de toplanmıştır — sadece o hücreyi düzenleyerek deneyi kontrol edebilirsin. Sonuçlar `./results/C3_CHA_*` altına kaydedilir.

In [ ]:
# ─── HÜCRE 1: Setup ────────────────────────────────────────────────────
# Colab tespit + pip install + Drive mount (opsiyonel) + arf_lib import
import os, sys, importlib
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    # Colab'da arf_lib.py'nin yerini bul — varsayım:
    # repo /content/CALMA/standalone_notebooks/ altında klonlu
    REPO_BASE = "/content/CALMA"
    NB_DIR = f"{REPO_BASE}/standalone_notebooks"
    if not os.path.exists(NB_DIR):
        !git clone https://github.com/hakanskn/CALMA.git {REPO_BASE} 2>&1 | tail -5
    sys.path.insert(0, NB_DIR)

    # Drive mount (sonuçları Drive'a yedeklemek istersen PARAMS'ta results_root değiştir)
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print("Drive mount skipped:", e)

    !pip install -q transformers==4.41.0 datasets==2.19.0 tokenizers==0.19.0 accelerate==0.30.0 matplotlib seaborn
else:
    # Lokal: arf_lib.py notebook ile aynı klasörde
    NB_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, NB_DIR)

import arf_lib
importlib.reload(arf_lib)
print(f"arf_lib loaded from: {arf_lib.__file__}")
print(f"GPU info: {arf_lib.gpu_info()}")

In [ ]:
# ─── HÜCRE 2: PARAMETRELER ─────────────────────────────────────────────
PARAMS = {
    "method_name":   "C3_CHA",
    "run_name":      "wt2_seed42",
    "seed":          42,
    "results_root":  "./results",

    "model_name":    "gpt2",
    "tokenizer_name": "gpt2",

    "dataset":       "wikitext2",
    "max_seq_len":   512,
    "batch_size":    16,
    "num_workers":   2,

    "learning_rate": 5e-5,
    "num_epochs":    3,
    "warmup_steps":  100,
    "grad_clip":     1.0,
    "weight_decay":  0.01,

    "limit_train_batches": 100,
    "limit_eval_batches":  50,

    "log_every_n_steps":        25,
    "save_checkpoints":         False,
    "checkpoint_every_n_steps": 500,
    "keep_last_n_checkpoints":  3,

    # ─ Method-specific (CHA)
    "cha_hebb_lr":        0.01,
    "cha_neg_ratio":      1.0,
    "cha_temperature":    1.0,
    "cha_clamp_strength": 0.1,
}

In [ ]:
# ─── HÜCRE 3: Imports ──────────────────────────────────────────────────
import math, time, json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from arf_lib import (
    seed_everything, get_device, gpu_info, gpu_peak_mb,
    GPT2Wrapper, BaseRouterAttention,
    StandaloneTrainer, load_text_dataloaders,
    run_pipeline, save_results, append_to_index, make_plots,
)

seed_everything(PARAMS["seed"])
print(f"Seed set: {PARAMS['seed']}")
print(f"Device: {get_device()}")

In [ ]:
# ─── HÜCRE 4: Method — Contrastive Hebbian Attention ───────────────────
class ContrastiveHebbianAttention(BaseRouterAttention):
    def __init__(self, gpt2_config, method_params):
        super().__init__(gpt2_config, method_params)
        mp = method_params
        self.hebb_lr = float(mp.get("cha_hebb_lr", 0.01))
        self.neg_ratio = float(mp.get("cha_neg_ratio", 1.0))
        self.temperature = float(mp.get("cha_temperature", 1.0))
        self.clamp_strength = float(mp.get("cha_clamp_strength", 0.1))

    def _scores(self, q, k):
        return torch.matmul(q, k.transpose(-1, -2)) * self.scale

    def _attend(self, q, k, v, hidden_states, attention_mask):
        s_minus = self._scores(q, k)
        noise = torch.randn_like(hidden_states) * self.clamp_strength * hidden_states.std()
        clamped = hidden_states + noise
        qkv_p = self.c_attn(clamped)
        qp, kp, _ = qkv_p.split(self.embed_dim, dim=-1)
        qp = self._split_heads(qp); kp = self._split_heads(kp)
        s_plus = self._scores(qp, kp)

        delta = (s_plus - self.neg_ratio * s_minus) / self.temperature
        causal = self._causal_mask(q.size(-2), k.size(-2), q.device)
        delta = delta.masked_fill(~causal, float("-inf"))
        if attention_mask is not None:
            delta = delta + attention_mask

        weights = F.softmax(F.relu(delta), dim=-1).clamp(min=1e-9)
        weights = weights / weights.sum(-1, keepdim=True)
        weights = self.attn_dropout(weights)
        out = torch.matmul(weights, v)

        if self.training:
            with torch.no_grad():
                hp = clamped.mean(0)
                hm = hidden_states.mean(0)
                dW = self.hebb_lr * (hp.T @ hp - hm.T @ hm) / max(1, hp.size(0))
                kw = self.c_attn.weight.data[self.embed_dim:2*self.embed_dim]
                rows = min(kw.size(0), dW.size(0))
                cols = min(kw.size(1), dW.size(1))
                kw[:rows, :cols] += dW[:rows, :cols]
                self.c_attn.weight.data[self.embed_dim:2*self.embed_dim] = kw

        self._last_stats = {"delta_mean": float((s_plus - s_minus).abs().mean().item())}
        return out, weights

print("ContrastiveHebbianAttention tanımlandı.")

In [ ]:
def build_model_and_adapter(params):
    wrapper = GPT2Wrapper(params["model_name"])
    wrapper.inject_attention(ContrastiveHebbianAttention, params)
    print(f"CHA injected → {sum(p.numel() for p in wrapper.parameters()):,} params")
    return wrapper, None

In [ ]:
# ─── HÜCRE 6: Run ──────────────────────────────────────────────────────
# Tek satır: model + opsiyonel adapter inşa et, pipeline çalıştır
model, adapter = build_model_and_adapter(PARAMS)
run_dir, result = run_pipeline(model, PARAMS, adapter=adapter)
print("\n" + "=" * 60)
print(f"RUN DIR: {run_dir}")
print(f"Final test PPL: {result['final_metrics']['test_ppl']:.4f}")
print(f"Final test BPC: {result['final_metrics']['test_bpc']:.4f}")
print(f"Duration: {result['duration_seconds']:.1f}s")
print("=" * 60)

In [ ]:
# ─── HÜCRE 7: Display plots (inline) ───────────────────────────────────
from IPython.display import Image, display
import os
plots_dir = run_dir / "plots"
if plots_dir.exists():
    for p in sorted(plots_dir.glob("*.png")):
        print(f"📈 {p.name}")
        display(Image(filename=str(p)))
else:
    print("Plots not generated.")
print(f"\nAll outputs in: {run_dir}")
print(f"Index file: {Path(PARAMS['results_root']) / '_index.csv'}")